# Milestone 3 — Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target check-in:** Pertemuan 8  
**Nama:** Luqmanul Hakim  
**NIM:** 202310370311141  
**Username GitHub:** luqmanulkim

Notebook ini memakai hasil Milestone 2 di `data/processed/dataset_bersih.parquet`. Dataset berisi pertanyaan, tanggal, metadata sumber, dan dua respons kandidat. Karena data tidak menyediakan koordinat geografis, bagian spasial dibahas sebagai keterbatasan; visualisasi keenam menggunakan hubungan multi-dimensi antar panjang teks.

## Bagian 1: Inisialisasi dan Memuat Data Bersih

Path dibuat relatif terhadap root proyek agar notebook dapat dijalankan dari root atau folder `notebooks/`.

In [ ]:
from pathlib import Path
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} tidak ditemukan. Jalankan notebooks/02_data_cleaning.ipynb terlebih dahulu."
    )

df = pl.read_parquet(DATA_PATH).with_columns(
    pl.col("date").str.to_date(format="%Y/%m/%d", strict=False).alias("tanggal"),
    pl.col("question").str.len_chars().alias("panjang_question"),
    pl.col("response_j").str.len_chars().alias("panjang_response_j"),
    pl.col("response_k").str.len_chars().alias("panjang_response_k"),
)
print(f"Data loaded: {df.height:,} baris x {df.width} kolom")
print(f"Periode: {df['tanggal'].min()} sampai {df['tanggal'].max()}")
df.head(5)

## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Tanggal dikelompokkan per bulan memakai `group_by_dynamic`. `rolling_mean` menambahkan rerata bergerak tiga bulan untuk membantu melihat tren tanpa menghapus variasi bulanan.

In [ ]:
df_valid_date = df.filter(pl.col("tanggal").is_not_null()).sort("tanggal")
monthly = (
    df_valid_date.group_by_dynamic("tanggal", every="1mo")
    .agg(pl.len().alias("jumlah_pertanyaan"))
    .sort("tanggal")
    .with_columns(
        pl.col("jumlah_pertanyaan").rolling_mean(window_size=3, min_samples=1)
        .alias("rerata_bergerak_3_bulan")
    )
)

# Fitur kategori untuk pola kalender dan asal sumber metadata.
features = df_valid_date.with_columns([
    pl.col("tanggal").dt.month().alias("bulan"),
    pl.col("tanggal").dt.weekday().alias("hari_angka"),
    pl.col("tanggal").dt.strftime("%A").alias("hari"),
    pl.col("metadata").list.first().str.extract(r"https?://([^/]+)", 1)
      .fill_null("Tidak diketahui").alias("domain_sumber"),
    ((pl.col("panjang_response_j") + pl.col("panjang_response_k")) / 2)
      .alias("rata_panjang_respons"),
])
print(f"Tanggal valid: {df_valid_date.height:,} dari {df.height:,}")
monthly.head()

## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly / Altair)

Grafik dapat diperbesar, diarahkan untuk melihat nilai, dan (untuk deret waktu) digeser melalui range slider.

### Visualisasi 1: Tren Deret Waktu dengan Range Slider



In [ ]:
# 1. Tren bulanan dan rerata bergerak tiga bulan
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=monthly["tanggal"], y=monthly["jumlah_pertanyaan"],
                          mode="lines", name="Jumlah bulanan"))
fig1.add_trace(go.Scatter(x=monthly["tanggal"], y=monthly["rerata_bergerak_3_bulan"],
                          mode="lines", name="Rerata bergerak 3 bulan"))
fig1.update_layout(title="Jumlah pertanyaan per bulan", xaxis_title="Bulan",
                   yaxis_title="Jumlah baris", hovermode="x unified")
fig1.update_xaxes(rangeslider_visible=True)
fig1.show()

### Visualisasi 2: Distribusi Variabel Utama dan Outlier



In [ ]:
# 2. Distribusi panjang pertanyaan dan respons; sumbu log membantu membaca ekor panjang.
long_text = features.select([
    pl.col("panjang_question").alias("Panjang pertanyaan"),
    pl.col("panjang_response_j").alias("Respons J"),
    pl.col("panjang_response_k").alias("Respons K"),
]).unpivot(variable_name="jenis_teks", value_name="jumlah_karakter")
# Batasi tampilan sampai persentil ke-99 agar mayoritas distribusi terbaca; data asli tetap utuh.
batas_p99 = long_text["jumlah_karakter"].quantile(0.99)
fig2 = px.box(long_text.filter(pl.col("jumlah_karakter") <= batas_p99).to_pandas(),
              x="jenis_teks", y="jumlah_karakter", color="jenis_teks",
              title="Distribusi panjang teks (hingga persentil ke-99)", points="outliers")
fig2.update_yaxes(type="log", title="Jumlah karakter (skala log)")
fig2.show()

### Visualisasi 3: Korelasi Antar-Variabel Numerik



In [ ]:
# 3. Korelasi fitur numerik. Korelasi panjang bukan ukuran kualitas jawaban.
numeric = features.select([
    "panjang_question", "panjang_response_j", "panjang_response_k"
]).to_pandas()
corr = numeric.corr(numeric_only=True)
fig3 = px.imshow(corr, text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
                 title="Korelasi antar fitur numerik")
fig3.show()

### Visualisasi 4: Komposisi Kategori Utama



In [ ]:
# 4. Komposisi 15 domain sumber yang paling sering muncul pada metadata.
top_domains = (features.group_by("domain_sumber").len(name="jumlah")
               .sort("jumlah", descending=True).head(15).sort("jumlah"))
fig4 = px.bar(top_domains.to_pandas(), x="jumlah", y="domain_sumber", orientation="h",
              title="15 domain sumber paling sering tercatat di metadata",
              labels={"jumlah":"Jumlah baris", "domain_sumber":"Domain"},
              color="jumlah", color_continuous_scale="Blues")
fig4.show()

### Visualisasi 5: Analisis Siklus Musiman dan Perbandingan Kelompok



In [ ]:
# 5. Pola kalender: jumlah pengamatan menurut bulan dan hari dalam minggu.
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
season = (features.group_by(["bulan", "hari"]).len(name="jumlah")
          .with_columns(pl.col("hari").cast(pl.Enum(weekday_order)))
          .sort(["bulan", "hari"]))
fig5 = px.density_heatmap(season.to_pandas(), x="bulan", y="hari", z="jumlah",
                          histfunc="sum", nbinsx=12, color_continuous_scale="Viridis",
                          title="Kepadatan baris menurut bulan dan hari dalam minggu",
                          labels={"bulan":"Bulan kalender", "hari":"Hari", "jumlah":"Jumlah"})
fig5.update_xaxes(dtick=1)
fig5.show()

### Visualisasi 6: Analisis Multi-Dimensi (Pengganti Spasial)



In [ ]:
# 6. Hubungan multi-dimensi teks. Sampel tetap dengan seed agar plot responsif dan reprodusibel.
scatter_data = (features.select(["panjang_question", "rata_panjang_respons", "bulan", "is_question_extreme"])
                .sample(n=min(12000, features.height), with_replacement=False, seed=42)
                .to_pandas())
fig6 = px.scatter(scatter_data, x="panjang_question", y="rata_panjang_respons",
                  color="bulan", symbol="is_question_extreme", opacity=0.55,
                  title="Panjang pertanyaan vs rata-rata panjang dua respons (sampel 12 ribu)",
                  labels={"panjang_question":"Karakter pertanyaan",
                          "rata_panjang_respons":"Rata-rata karakter respons",
                          "bulan":"Bulan", "is_question_extreme":"Pertanyaan ekstrem"},
                  hover_data=["is_question_extreme"])
fig6.show()

## Bagian 4: Rumusan 5 Insight Analitik Utama

Tabel berikut menghitung angka ringkasan dari data bersih. Baca panjang respons sebagai ukuran panjang teks, bukan penilaian mutu atau preferensi pengguna.

In [ ]:
summary = features.select([
    pl.len().alias("jumlah_baris"),
    pl.col("tanggal").min().alias("tanggal_awal"),
    pl.col("tanggal").max().alias("tanggal_akhir"),
    pl.col("qid").n_unique().alias("qid_unik"),
    pl.col("panjang_question").median().alias("median_panjang_pertanyaan"),
    pl.col("panjang_question").mean().round(1).alias("rata_panjang_pertanyaan"),
    pl.col("panjang_response_j").mean().round(1).alias("rata_panjang_respons_j"),
    pl.col("panjang_response_k").mean().round(1).alias("rata_panjang_respons_k"),
    pl.col("is_question_extreme").sum().alias("pertanyaan_ekstrem"),
]).row(0, named=True)
peak_month = monthly.sort("jumlah_pertanyaan", descending=True).row(0, named=True)
weekday_counts = features.group_by("hari").len(name="jumlah").sort("jumlah", descending=True)
peak_weekday = weekday_counts.row(0, named=True)
print("Ringkasan:", summary)
print("Bulan dengan jumlah tertinggi:", peak_month)
print("Hari dengan jumlah tertinggi:", peak_weekday)
print("Domain teratas:")
display(top_domains.sort("jumlah", descending=True))

### Lima insight berdasarkan hasil visualisasi

1. **Tren waktu dan musiman.** Dataset mencakup **1 Agustus 2008–4 Maret 2023**. Volume tertinggi terjadi pada **Maret 2011 (9.834 baris)**. Grafik bulanan dan heatmap kalender menunjukkan perubahan volume serta pola berulang. Perbedaan volume dapat mencerminkan cakupan sumber data, sehingga tidak langsung diartikan sebagai perubahan minat masyarakat.
2. **Distribusi dan anomali.** Median panjang pertanyaan lebih rendah daripada rata-ratanya; distribusi condong ke kanan karena sebagian pertanyaan jauh lebih panjang. **57.118** baris ditandai ekstrem pada Milestone 2 dan tetap dipertahankan. Gunakan persentil tinggi untuk meninjau kasus panjang tanpa menghapus konteks secara otomatis.
3. **Korelasi kunci.** Heatmap dan scatter plot mengukur hubungan antar panjang teks saja. Rata-rata panjang respons J pada data ini sekitar **1.020 karakter**, sementara K sekitar **759 karakter**. Ini menunjukkan perbedaan panjang, bukan bahwa respons J lebih baik; evaluasi kualitas memerlukan rubrik atau penilaian tambahan.
4. **Segmen sumber dan kalender.** Grafik domain merangkum domain URL pertama dari metadata, sedangkan heatmap membandingkan bulan kalender dengan hari dalam minggu. Konsentrasi pada domain/hari tertentu dapat mencerminkan pola pengumpulan atau komposisi sumber. Sel `peak_month` dan `peak_weekday` memberi nilai tertinggi aktual saat notebook dijalankan.
5. **Rekomendasi praktis.** Saat membandingkan dua respons, kendalikan panjang pertanyaan dan periode sumber agar perbandingan lebih adil; tinjau sampel pertanyaan ekstrem secara manual; serta catat domain dan cakupan tanggal ketika membuat ringkasan. Dataset ini tidak memiliki koordinat geografis, jadi klaim spasial atau peta tidak dapat dibuat tanpa data lokasi tambahan.

**Catatan interpretasi:** observasi agregat tidak membuktikan sebab-akibat. Grafik ini mendeskripsikan dataset yang tersedia setelah pembersihan, bukan seluruh populasi percakapan.